In [1]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    lit,
    when
)

print("=" * 80)
print("GOLD DATA QUALITY CHECKS - MVP")
print("=" * 80)


# ---------------------------------------------------------
# 1. Expected row counts
# ---------------------------------------------------------

expected_counts = {
    "dim_learner": 3,
    "dim_topic": 10,
    "dim_content_type": 6,
    "dim_reference_source": 6,

    "fact_learning_interaction": 6,
    "fact_practice_attempt": 5,
    "fact_learning_session": 3,
    "fact_learning_feedback": 11,
    "fact_ai_insight_validation": 9,
    "fact_learner_concept_state": 12,

    "agg_learner_overview_daily": 9,
    "agg_concept_weakness_daily": 17,
    "agg_learning_progress_daily": 9,
    "agg_illusion_of_learning": 3,

    "ml_learning_difficulty_features": 3
}

print("\n1. ROW COUNT CHECKS")

count_results = []

for table_name, expected_count in expected_counts.items():
    full_table_name = f"demo.gold.{table_name}"

    actual_count = spark.table(full_table_name).count()

    status = (
        "PASS"
        if actual_count == expected_count
        else "FAIL"
    )

    count_results.append(
        (
            table_name,
            expected_count,
            actual_count,
            status
        )
    )

spark.createDataFrame(
    count_results,
    [
        "table_name",
        "expected_count",
        "actual_count",
        "status"
    ]
).show(
    truncate=False
)


# ---------------------------------------------------------
# 2. Grain / duplicate checks
# ---------------------------------------------------------

print("\n2. GRAIN AND DUPLICATE CHECKS")

grain_checks = {
    "dim_learner":
        ["user_key"],

    "dim_topic":
        ["topic_key"],

    "dim_content_type":
        ["content_type_key"],

    "dim_reference_source":
        ["reference_key"],

    "fact_learning_interaction":
        ["interaction_key"],

    "fact_practice_attempt":
        ["attempt_key"],

    "fact_learning_session":
        ["session_id"],

    "fact_learning_feedback":
        ["feedback_key"],

    "fact_ai_insight_validation":
        ["validation_key"],

    "fact_learner_concept_state":
        [
            "user_key",
            "topic_key",
            "state_version"
        ],

    "agg_learner_overview_daily":
        [
            "user_key",
            "date"
        ],

    "agg_concept_weakness_daily":
        [
            "user_key",
            "topic_key",
            "date"
        ],

    "agg_learning_progress_daily":
        [
            "user_key",
            "date"
        ],

    "agg_illusion_of_learning":
        [
            "user_key",
            "topic_key",
            "session_id"
        ],

    "ml_learning_difficulty_features":
        [
            "user_key",
            "topic_key",
            "session_id"
        ]
}

grain_results = []

for table_name, grain_columns in grain_checks.items():
    df = spark.table(
        f"demo.gold.{table_name}"
    )

    total_rows = df.count()

    distinct_rows = (
        df
        .select(*grain_columns)
        .distinct()
        .count()
    )

    duplicate_rows = total_rows - distinct_rows

    status = (
        "PASS"
        if duplicate_rows == 0
        else "FAIL"
    )

    grain_results.append(
        (
            table_name,
            ", ".join(grain_columns),
            total_rows,
            distinct_rows,
            duplicate_rows,
            status
        )
    )

spark.createDataFrame(
    grain_results,
    [
        "table_name",
        "grain",
        "total_rows",
        "distinct_grain_rows",
        "duplicate_rows",
        "status"
    ]
).show(
    truncate=False
)


# ---------------------------------------------------------
# 3. Required-key null checks
# ---------------------------------------------------------

print("\n3. REQUIRED KEY NULL CHECKS")

required_key_checks = {
    "dim_learner":
        ["user_key", "user_id"],

    "dim_topic":
        ["topic_key", "topic_id"],

    "dim_content_type":
        ["content_type_key", "content_type_id"],

    "dim_reference_source":
        ["reference_key", "reference_id"],

    "fact_learning_interaction":
        [
            "interaction_key",
            "event_id",
            "user_key",
            "session_id"
        ],

    "fact_practice_attempt":
        [
            "attempt_key",
            "attempt_id",
            "event_id",
            "user_key",
            "topic_key",
            "session_id"
        ],

    "fact_learning_session":
        [
            "session_id",
            "user_key",
            "session_start_time"
        ],

    "fact_learning_feedback":
        [
            "feedback_key",
            "user_key",
            "feedback_stage",
            "feedback_time"
        ],

    "fact_ai_insight_validation":
        [
            "validation_key",
            "validation_id",
            "insight_id",
            "event_id",
            "user_key",
            "topic_key",
            "reference_key"
        ],

    "fact_learner_concept_state":
        [
            "concept_state_key",
            "user_key",
            "topic_key",
            "state_version",
            "last_evidence_time"
        ],

    "agg_learner_overview_daily":
        ["user_key", "date"],

    "agg_concept_weakness_daily":
        ["user_key", "topic_key", "date"],

    "agg_learning_progress_daily":
        ["user_key", "date"],

    "agg_illusion_of_learning":
        ["user_key", "topic_key", "session_id"],

    "ml_learning_difficulty_features":
        ["user_key", "topic_key", "session_id"]
}

null_results = []

for table_name, required_columns in required_key_checks.items():
    df = spark.table(
        f"demo.gold.{table_name}"
    )

    null_condition = None

    for column_name in required_columns:
        current_condition = col(column_name).isNull()

        if null_condition is None:
            null_condition = current_condition
        else:
            null_condition = (
                null_condition
                | current_condition
            )

    invalid_rows = (
        df
        .filter(null_condition)
        .count()
    )

    status = (
        "PASS"
        if invalid_rows == 0
        else "FAIL"
    )

    null_results.append(
        (
            table_name,
            ", ".join(required_columns),
            invalid_rows,
            status
        )
    )

spark.createDataFrame(
    null_results,
    [
        "table_name",
        "required_columns",
        "rows_with_missing_required_values",
        "status"
    ]
).show(
    truncate=False
)


# ---------------------------------------------------------
# 4. Foreign-key checks
# ---------------------------------------------------------

print("\n4. FOREIGN KEY CHECKS")

dim_learner_df = spark.table(
    "demo.gold.dim_learner"
)

dim_topic_df = spark.table(
    "demo.gold.dim_topic"
)

dim_reference_df = spark.table(
    "demo.gold.dim_reference_source"
)

fact_tables_with_user = [
    "fact_learning_interaction",
    "fact_practice_attempt",
    "fact_learning_session",
    "fact_learning_feedback",
    "fact_ai_insight_validation",
    "fact_learner_concept_state",
    "agg_learner_overview_daily",
    "agg_concept_weakness_daily",
    "agg_learning_progress_daily",
    "agg_illusion_of_learning",
    "ml_learning_difficulty_features"
]

foreign_key_results = []

for table_name in fact_tables_with_user:
    df = spark.table(
        f"demo.gold.{table_name}"
    )

    missing_users = (
        df
        .select("user_key")
        .filter(col("user_key").isNotNull())
        .distinct()
        .join(
            dim_learner_df.select("user_key"),
            ["user_key"],
            "left_anti"
        )
        .count()
    )

    foreign_key_results.append(
        (
            table_name,
            "user_key -> dim_learner",
            missing_users,
            "PASS" if missing_users == 0 else "FAIL"
        )
    )

fact_tables_with_topic = [
    "fact_practice_attempt",
    "fact_ai_insight_validation",
    "fact_learner_concept_state",
    "agg_concept_weakness_daily",
    "agg_illusion_of_learning",
    "ml_learning_difficulty_features"
]

for table_name in fact_tables_with_topic:
    df = spark.table(
        f"demo.gold.{table_name}"
    )

    missing_topics = (
        df
        .select("topic_key")
        .filter(col("topic_key").isNotNull())
        .distinct()
        .join(
            dim_topic_df.select("topic_key"),
            ["topic_key"],
            "left_anti"
        )
        .count()
    )

    foreign_key_results.append(
        (
            table_name,
            "topic_key -> dim_topic",
            missing_topics,
            "PASS" if missing_topics == 0 else "FAIL"
        )
    )

missing_references = (
    spark.table(
        "demo.gold.fact_ai_insight_validation"
    )
    .select("reference_key")
    .filter(col("reference_key").isNotNull())
    .distinct()
    .join(
        dim_reference_df.select("reference_key"),
        ["reference_key"],
        "left_anti"
    )
    .count()
)

foreign_key_results.append(
    (
        "fact_ai_insight_validation",
        "reference_key -> dim_reference_source",
        missing_references,
        "PASS" if missing_references == 0 else "FAIL"
    )
)

spark.createDataFrame(
    foreign_key_results,
    [
        "table_name",
        "relationship",
        "missing_foreign_keys",
        "status"
    ]
).show(
    truncate=False
)


# ---------------------------------------------------------
# 5. Basic score and range checks
# ---------------------------------------------------------

print("\n5. SCORE AND RANGE CHECKS")

range_results = []

practice_invalid = (
    spark.table(
        "demo.gold.fact_practice_attempt"
    )
    .filter(
        (col("score") < 0)
        | (col("score") > 1)
        | (col("hints_used") < 0)
        | (col("attempt_duration_seconds") < 0)
        | (col("attempt_number") < 1)
    )
    .count()
)

range_results.append(
    (
        "fact_practice_attempt",
        practice_invalid,
        "PASS" if practice_invalid == 0 else "FAIL"
    )
)

validation_invalid = (
    spark.table(
        "demo.gold.fact_ai_insight_validation"
    )
    .filter(
        (col("extraction_confidence") < 0)
        | (col("extraction_confidence") > 1)
        | (col("semantic_match_score") < 0)
        | (col("semantic_match_score") > 1)
        | (col("reliability_score") < 0)
        | (col("reliability_score") > 1)
    )
    .count()
)

range_results.append(
    (
        "fact_ai_insight_validation",
        validation_invalid,
        "PASS" if validation_invalid == 0 else "FAIL"
    )
)

state_invalid = (
    spark.table(
        "demo.gold.fact_learner_concept_state"
    )
    .filter(
        (
            col("mastery_score").isNotNull()
            & (
                (col("mastery_score") < 0)
                | (col("mastery_score") > 1)
            )
        )
        |
        (
            col("difficulty_score").isNotNull()
            & (
                (col("difficulty_score") < 0)
                | (col("difficulty_score") > 1)
            )
        )
        |
        (
            col("confidence_score").isNotNull()
            & (
                (col("confidence_score") < 0)
                | (col("confidence_score") > 1)
            )
        )
        |
        (
            col("struggle_risk_score").isNotNull()
            & (
                (col("struggle_risk_score") < 0)
                | (col("struggle_risk_score") > 1)
            )
        )
        |
        (col("repeated_mistake_count") < 0)
        |
        (col("evidence_count") < 1)
    )
    .count()
)

range_results.append(
    (
        "fact_learner_concept_state",
        state_invalid,
        "PASS" if state_invalid == 0 else "FAIL"
    )
)

illusion_invalid = (
    spark.table(
        "demo.gold.agg_illusion_of_learning"
    )
    .filter(
        (col("practice_score") < 0)
        | (col("practice_score") > 1)
        | (col("illusion_gap_score") < -1)
        | (col("illusion_gap_score") > 1)
    )
    .count()
)

range_results.append(
    (
        "agg_illusion_of_learning",
        illusion_invalid,
        "PASS" if illusion_invalid == 0 else "FAIL"
    )
)

spark.createDataFrame(
    range_results,
    [
        "table_name",
        "invalid_range_rows",
        "status"
    ]
).show(
    truncate=False
)


# ---------------------------------------------------------
# 6. Final MVP summary
# ---------------------------------------------------------

failed_count_checks = sum(
    1
    for row in count_results
    if row[3] == "FAIL"
)

failed_grain_checks = sum(
    1
    for row in grain_results
    if row[5] == "FAIL"
)

failed_null_checks = sum(
    1
    for row in null_results
    if row[3] == "FAIL"
)

failed_fk_checks = sum(
    1
    for row in foreign_key_results
    if row[3] == "FAIL"
)

failed_range_checks = sum(
    1
    for row in range_results
    if row[2] == "FAIL"
)

total_failed_checks = (
    failed_count_checks
    + failed_grain_checks
    + failed_null_checks
    + failed_fk_checks
    + failed_range_checks
)

print("=" * 80)

if total_failed_checks == 0:
    print("FINAL GOLD QUALITY STATUS: PASS")
else:
    print("FINAL GOLD QUALITY STATUS: FAIL")
    print(
        "Failed check groups:",
        total_failed_checks
    )

print("=" * 80)

GOLD DATA QUALITY CHECKS - MVP

1. ROW COUNT CHECKS


+-------------------------------+--------------+------------+------+
|table_name                     |expected_count|actual_count|status|
+-------------------------------+--------------+------------+------+
|dim_learner                    |3             |3           |PASS  |
|dim_topic                      |10            |10          |PASS  |
|dim_content_type               |6             |6           |PASS  |
|dim_reference_source           |6             |6           |PASS  |
|fact_learning_interaction      |6             |6           |PASS  |
|fact_practice_attempt          |5             |5           |PASS  |
|fact_learning_session          |3             |3           |PASS  |
|fact_learning_feedback         |11            |11          |PASS  |
|fact_ai_insight_validation     |9             |9           |PASS  |
|fact_learner_concept_state     |12            |12          |PASS  |
|agg_learner_overview_daily     |9             |9           |PASS  |
|agg_concept_weakness_daily     |1

+-------------------------------+----------------------------------+----------+-------------------+--------------+------+
|table_name                     |grain                             |total_rows|distinct_grain_rows|duplicate_rows|status|
+-------------------------------+----------------------------------+----------+-------------------+--------------+------+
|dim_learner                    |user_key                          |3         |3                  |0             |PASS  |
|dim_topic                      |topic_key                         |10        |10                 |0             |PASS  |
|dim_content_type               |content_type_key                  |6         |6                  |0             |PASS  |
|dim_reference_source           |reference_key                     |6         |6                  |0             |PASS  |
|fact_learning_interaction      |interaction_key                   |6         |6                  |0             |PASS  |
|fact_practice_attempt  